In [1]:
# Data and Math Libraries
import os
os.environ.setdefault('POLARS_MAX_THREADS', '1')  # avoids a torch/polars native-thread deadlock on macOS
import polars as pl
import numpy as np
from datetime import datetime, timedelta
import random

#Machine Learning Libraries
import torch
import torch.nn as nn 
import torch.optim as optim
import research 
import models

#visualization 
import altair as alt

#data sources
import binance

In [2]:
print(research.__file__)
print([x for x in dir(research) if not x.startswith('_')])

/Users/arezziorietti/Documents/ML Projects/research.py
['BINANCE_TRADE_CSV_COLUMNS', 'BINANCE_VISION_TRADES_URL', 'Dict', 'List', 'OHLC_AGGS', 'Optional', 'Path', 'SEED', 'Tuple', 'Union', 'add_compounding_trades', 'add_equity_curve', 'add_lags', 'add_log_return_features', 'add_model_predictions', 'add_trade_log_returns', 'add_tx_fee', 'add_tx_fees', 'add_tx_fees_log', 'altair', 'auto_reg_corr_matrx', 'batch_train_reg', 'benchmark_linear_models', 'benchmark_reg_model', 'buy_and_hold_performance', 'compare_ts_corr', 'datetime', 'download_trades', 'eval_model_performance', 'get_linear_params', 'get_trade_files', 'init_weights', 'io', 'itertools', 'lag_col_names', 'lag_cols', 'learn_model_trade_pnl', 'learn_model_trades', 'load_ohlc_timeseries', 'load_ohlc_timeseries_range', 'load_timeseries', 'load_timeseries_range', 'log_return', 'log_return_col', 'log_returns_col', 'math', 'model_trade_results', 'models_module', 'nn', 'np', 'npt', 'ohlc_timeseries', 'optim', 'os', 'pl', 'plot', 'plot_c

In [3]:
research.set_seed(42)

In [4]:
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_cols(-1) 

polars.config.Config

In [5]:
sym = 'BTCUSDT'

hist_data_window = 7 * 4 * 6

time_interval = '1h'

max_lags = 4

forecast_horizon = 1 

annualized_rate = research.sharpe_annualization_factor(time_interval, 365, 24)

# Part 1: BTCUSDT Research

## Download & Load Price Data

In [6]:
# Download hist_data_window days of trade data (skips days already cached)
binance.download_trades(sym, hist_data_window)

In [7]:
ts = research.load_ohlc_timeseries(sym, time_interval)
ts

Loading BTCUSDT:   0%|          | 0/169 [00:00<?, ?file/s]

Loading BTCUSDT:   1%|          | 1/169 [00:00<00:40,  4.17file/s]

Loading BTCUSDT:   1%|          | 2/169 [00:00<00:44,  3.77file/s]

Loading BTCUSDT:   2%|▏         | 3/169 [00:00<00:35,  4.70file/s]

Loading BTCUSDT:   2%|▏         | 4/169 [00:00<00:30,  5.38file/s]

...

Loading BTCUSDT:  99%|█████████▉| 167/169 [00:35<00:00,  7.00file/s]

Loading BTCUSDT:  99%|█████████▉| 168/169 [00:35<00:00,  6.18file/s]

Loading BTCUSDT: 100%|██████████| 169/169 [00:35<00:00,  5.97file/s]

Loading BTCUSDT: 100%|██████████| 169/169 [00:35<00:00,  4.74file/s]

datetime,open,high,low,close
datetime[μs],f64,f64,f64,f64
2026-01-20 00:00:00,92589.6,92845.0,92320.2,92676.1
2026-01-20 01:00:00,92676.1,92840.0,92479.1,92688.7
2026-01-20 02:00:00,92688.8,92693.4,92330.0,92518.9
2026-01-20 03:00:00,92519.0,92646.2,92200.0,92397.0
2026-01-20 04:00:00,92396.9,92440.1,91900.0,92026.2
…,…,…,…,…
2026-07-07 19:00:00,63653.6,63873.2,63351.4,63794.9
2026-07-07 20:00:00,63794.9,63874.6,63588.5,63668.3
2026-07-07 21:00:00,63668.4,63839.9,63264.0,63399.5


In [8]:
research.plot_static_timeseries(ts, sym, 'close', time_interval)

/Users/arezziorietti/Documents/ML Projects/research.py:542: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Feature Engineering

Build the target (future log return) and auto-regressive lag features.

In [9]:
target = 'close_log_return'
ts = research.add_log_return_features(ts, 'close', forecast_horizon, max_no_lags=max_lags)
ts = ts.drop_nulls()
ts

datetime,open,high,low,close,close_log_return,close_log_return_lag_1,close_log_return_lag_2,close_log_return_lag_3,close_log_return_lag_4
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-01-20 05:00:00,92026.3,92042.1,91235.5,91590.1,-0.00475,-0.004021,-0.001318,-0.001834,0.000136
2026-01-20 06:00:00,91590.1,91603.8,90723.3,90894.5,-0.007624,-0.00475,-0.004021,-0.001318,-0.001834
2026-01-20 07:00:00,90894.5,91268.8,90819.5,91136.1,0.002655,-0.007624,-0.00475,-0.004021,-0.001318
2026-01-20 08:00:00,91136.1,91222.0,90622.2,91078.9,-0.000628,0.002655,-0.007624,-0.00475,-0.004021
2026-01-20 09:00:00,91078.9,91078.9,90860.8,91002.9,-0.000835,-0.000628,0.002655,-0.007624,-0.00475
…,…,…,…,…,…,…,…,…,…
2026-07-07 19:00:00,63653.6,63873.2,63351.4,63794.9,0.002214,-0.007994,0.003158,0.00082,0.005861
2026-07-07 20:00:00,63794.9,63874.6,63588.5,63668.3,-0.001986,0.002214,-0.007994,0.003158,0.00082
2026-07-07 21:00:00,63668.4,63839.9,63264.0,63399.5,-0.004231,-0.001986,0.002214,-0.007994,0.003158


In [10]:
research.plot_distribution(ts, target, no_bins=100)

alt.Chart(...)

In [11]:
# Auto-correlation between the target and its own lags
research.auto_reg_corr_matrx(ts, target, max_lags)

close_log_return,close_log_return_lag_1,close_log_return_lag_2,close_log_return_lag_3,close_log_return_lag_4
f64,f64,f64,f64,f64
1.0,-0.008481,-0.019831,0.005264,-0.014129
-0.008481,1.0,-0.008386,-0.019874,0.005206
-0.019831,-0.008386,1.0,-0.008292,-0.019843
0.005264,-0.019874,-0.008292,1.0,-0.00837
-0.014129,0.005206,-0.019843,-0.00837,1.0


## Benchmark Linear Models

Try every combination of up to 3 lag features and rank by annualized Sharpe.

In [12]:
feature_pool = [f'{target}_lag_{i}' for i in range(1, max_lags + 1)]
btc_benchmark = research.benchmark_linear_models(
    ts, target, feature_pool, annualized_rate, max_no_features=3, loss=nn.L1Loss()
)
btc_benchmark

features,target,no_trades,win_rate,avg_win,avg_loss,best_trade,worst_trade,ev,std,total_log_return,compound_return,max_drawdown,equity_trough,equity_peak,sharpe,weights,biases
str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""close_log_return_lag_2,close_log_return_lag_4""","""close_log_return""",1013,0.530109,0.003668,-0.003214,0.049699,-0.022936,0.000434,0.005257,0.439939,1.552613,-0.087531,-0.05361,0.442769,7.731603,"""[-0.04638382 -0.02816361]""","""-1.1558848200365901e-05"""
"""close_log_return_lag_1,close_log_return_lag_2""","""close_log_return""",1013,0.529121,0.003589,-0.003305,0.049699,-0.023952,0.000343,0.005264,0.347159,1.415041,-0.083122,0.000136,0.349989,6.093207,"""[-0.02469324 -0.06081564]""","""-1.1738263310689945e-05"""
"""close_log_return_lag_2,close_log_return_lag_3""","""close_log_return""",1013,0.542942,0.003487,-0.003416,0.024953,-0.049699,0.000332,0.005265,0.336147,1.399545,-0.067633,0.002709,0.351881,5.899155,"""[-0.04075581 -0.02804954]""","""-3.80322421733581e-06"""
"""close_log_return_lag_1,close_log_return_lag_3,close_log_return_lag_4""","""close_log_return""",1013,0.538993,0.003482,-0.003423,0.024953,-0.049699,0.000299,0.005267,0.302559,1.353317,-0.131263,-0.073293,0.328924,5.3077,"""[-0.02887406 -0.02253534 -0.03189928]""","""-3.506228676997125e-05"""
"""close_log_return_lag_3,close_log_return_lag_4""","""close_log_return""",1013,0.521224,0.00356,-0.00334,0.024953,-0.049699,0.000256,0.005269,0.259543,1.296337,-0.104014,-0.034494,0.308145,4.55115,"""[-0.01292978 -0.03342096]""","""-1.731406882754527e-05"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""close_log_return_lag_1""","""close_log_return""",1013,0.538006,0.003323,-0.003608,0.049699,-0.023952,0.000121,0.005274,0.1225,1.130319,-0.157049,-0.003961,0.207991,2.146096,"""[-0.03995011]""","""-4.305058246245608e-05"""
"""close_log_return_lag_4""","""close_log_return""",1013,0.498519,0.00356,-0.00335,0.049699,-0.024953,0.000095,0.005274,0.095937,1.10069,-0.166285,-0.056645,0.214886,1.680565,"""[-0.03309952]""","""-2.5998548153438605e-05"""
"""close_log_return_lag_1,close_log_return_lag_2,close_log_return_lag_3""","""close_log_return""",1013,0.535044,0.00329,-0.003644,0.024953,-0.049699,0.000066,0.005275,0.066889,1.069177,-0.178602,0.001861,0.218239,1.171625,"""[-0.04644156 -0.04747161 -0.01680875]""","""-1.8258077034261078e-05"""


In [13]:
btc_best = btc_benchmark.row(0, named=True)
btc_best_features = btc_best['features'].split(',')
btc_best_features

['close_log_return_lag_2', 'close_log_return_lag_4']

## Best Model: Trade-Level Performance

In [14]:
btc_model = models.LinearModel(len(btc_best_features))
btc_model_trades = research.learn_model_trades(ts, btc_best_features, target, btc_model, loss=nn.L1Loss())
research.plot_column(btc_model_trades, 'equity_curve')

alt.Chart(...)

## Add Transaction Fees

In [15]:
maker_fee = 0.00045
taker_fee = 0.00045
btc_model_trades = research.add_tx_fees_log(btc_model_trades, maker_fee, taker_fee)
research.plot_column(btc_model_trades, 'equity_curve_net_taker')

alt.Chart(...)

## Save Model

In [16]:
torch.save(btc_model.state_dict(), 'BTCUSDT_model_weights.pth')
print(f"Saved BTCUSDT model with features {btc_best_features}")

Saved BTCUSDT model with features ['close_log_return_lag_2', 'close_log_return_lag_4']


## Out-of-Sample Validation (Walk-Forward)

The benchmark above picked the "best" feature combo by scanning every lag combination and choosing whichever had the highest Sharpe on a single holdout split. With that many combinations tried, a good-looking Sharpe can show up by luck alone (data snooping) — it doesn't prove the model has real predictive power.

This section fixes that: `walk_forward_eval` re-runs feature selection and training on rolling windows where each fold's test period was never seen during that fold's selection or training. The folds are then stitched into one continuous out-of-sample equity curve, which we compare against two baselines: buy-and-hold, and a random coin-flip trading signal (the null hypothesis of "no skill at all").

In [17]:
btc_wf_folds, btc_oos_y_test, btc_oos_y_hat = research.walk_forward_eval(
    ts, target, feature_pool, annualized_rate, n_folds=5, loss=nn.L1Loss()
)
btc_wf_folds

features,target,no_trades,win_rate,avg_win,avg_loss,best_trade,worst_trade,ev,std,total_log_return,compound_return,max_drawdown,equity_trough,equity_peak,sharpe,fold,no_test_obs
str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
"""close_log_return_lag_1,close_log_return_lag_2,close_log_return_lag_3""","""close_log_return""",405,0.555556,0.002401,-0.002769,0.013149,-0.016912,0.000103,0.003568,0.041821,1.042708,-0.058757,-0.013621,0.07016,2.708611,0,405
"""close_log_return_lag_2""","""close_log_return""",405,0.516049,0.002112,-0.002239,0.016465,-0.020395,0.000006,0.003335,0.002531,1.002534,-0.057864,-0.033703,0.054314,0.175379,1,405
"""close_log_return_lag_2,close_log_return_lag_4""","""close_log_return""",405,0.518519,0.002634,-0.0029,0.017843,-0.018718,-0.000031,0.004169,-0.012372,0.987704,-0.101138,-0.079405,0.021733,-0.685773,2,405
"""close_log_return_lag_2,close_log_return_lag_3""","""close_log_return""",405,0.520988,0.003757,-0.003632,0.02371,-0.022936,0.000218,0.005405,0.088142,1.092143,-0.067485,-0.020006,0.113039,3.768484,3,405
"""close_log_return_lag_2,close_log_return_lag_4""","""close_log_return""",406,0.534483,0.00352,-0.003154,0.049699,-0.014871,0.000413,0.005338,0.167819,1.182722,-0.072177,-0.00276,0.174373,7.247375,4,406


In [18]:
# Stitched-together out-of-sample performance across all folds
btc_oos_perf = research.eval_model_performance(btc_oos_y_test, btc_oos_y_hat, feature_pool, target, annualized_rate)
btc_oos_perf

{'features': 'close_log_return_lag_1,close_log_return_lag_2,close_log_return_lag_3,close_log_return_lag_4',
 'target': 'close_log_return',
 'no_trades': 2026,
 'win_rate': 0.5291214215202369,
 'avg_win': 0.0028836563780214926,
 'avg_loss': -0.0029385104119318915,
 'best_trade': 0.04969850182533264,
 'worst_trade': -0.022935960441827774,
 'ev': 0.0001421227562961576,
 'std': 0.004447606857866049,
 'total_log_return': 0.28794071078300476,
 'compound_return': np.float64(1.3336782290413343),
 'max_drawdown': -0.13118776679039001,
 'equity_trough': -0.03505219146609306,
 'equity_peak': 0.2944944500923157,
 'sharpe': np.float64(2.990813163078893)}

### Baselines: Buy & Hold and a Random-Signal Null Distribution

In [19]:
btc_bh_perf = research.buy_and_hold_performance(btc_oos_y_test, annualized_rate)
btc_bh_perf

{'strategy': 'buy_and_hold',
 'no_periods': 2026,
 'total_log_return': -0.17530666291713715,
 'compound_return': 0.8391996026039124,
 'std': 0.004447938408702612,
 'max_drawdown': -0.3479048013687134,
 'sharpe': -1.8207580714737432}

In [20]:
# Null distribution: 1000 random +1/-1 signals against the SAME realized OOS returns.
# If our model has no real edge, its Sharpe should look like a typical draw from this distribution.
btc_null_sharpes = research.random_signal_sharpes(btc_oos_y_test, annualized_rate, n_trials=1000, seed=42)
btc_p_value = research.sharpe_p_value(btc_oos_perf['sharpe'], btc_null_sharpes)

print(f"BTCUSDT OOS model Sharpe:      {btc_oos_perf['sharpe']:.2f}")
print(f"BTCUSDT Buy & Hold Sharpe:     {btc_bh_perf['sharpe']:.2f}")
print(f"Random-signal null Sharpe:     mean={btc_null_sharpes.mean():.2f}, std={btc_null_sharpes.std():.2f}")
print(f"P(random signal Sharpe >= model OOS Sharpe): {btc_p_value:.3f}")

BTCUSDT OOS model Sharpe:      2.99
BTCUSDT Buy & Hold Sharpe:     -1.82
Random-signal null Sharpe:     mean=-0.07, std=2.10
P(random signal Sharpe >= model OOS Sharpe): 0.068


In [21]:
import matplotlib.pyplot as plt

btc_oos_trades = research.model_trade_results(btc_oos_y_test, btc_oos_y_hat)
btc_bh_equity_curve = np.cumsum(btc_oos_y_test.numpy().flatten())

plt.figure(figsize=(12, 6))
plt.plot(btc_oos_trades['equity_curve'], label='Model (walk-forward OOS)')
plt.plot(btc_bh_equity_curve, label='Buy & Hold')
plt.title('BTCUSDT: Out-of-Sample Equity Curve vs Buy & Hold')
plt.xlabel('Period (stitched across folds)')
plt.ylabel('Cumulative Log Return')
plt.legend()
plt.tight_layout()
plt.show()

/var/folders/69/1t3g51kj5y96vspv1zh9f1x40000gn/T/ipykernel_30504/2974186181.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Part 2: ETHUSDT Research

Repeat the exact same research pipeline on a second asset (ETHUSDT) so we're not overfitting our conclusions to a single symbol.

## Download & Load Price Data

In [22]:
sym = 'ETHUSDT'
binance.download_trades(sym, hist_data_window)

In [23]:
eth_ts = research.load_ohlc_timeseries(sym, time_interval)
eth_ts

Loading ETHUSDT:   0%|          | 0/168 [00:00<?, ?file/s]

Loading ETHUSDT:   1%|          | 1/168 [00:00<01:21,  2.05file/s]

Loading ETHUSDT:   1%|          | 2/168 [00:00<01:05,  2.54file/s]

Loading ETHUSDT:   2%|▏         | 3/168 [00:01<00:57,  2.88file/s]

Loading ETHUSDT:   3%|▎         | 5/168 [00:01<00:38,  4.21file/s]

...

Loading ETHUSDT:  99%|█████████▉| 166/168 [00:51<00:00,  5.50file/s]

Loading ETHUSDT:  99%|█████████▉| 167/168 [00:51<00:00,  4.88file/s]

Loading ETHUSDT: 100%|██████████| 168/168 [00:51<00:00,  4.63file/s]

Loading ETHUSDT: 100%|██████████| 168/168 [00:51<00:00,  3.25file/s]

datetime,open,high,low,close
datetime[μs],f64,f64,f64,f64
2026-01-21 00:00:00,2938.25,2968.0,2920.08,2958.32
2026-01-21 01:00:00,2958.31,2973.06,2953.62,2963.66
2026-01-21 02:00:00,2963.66,2973.75,2956.22,2960.65
2026-01-21 03:00:00,2960.65,2983.7,2959.79,2980.27
2026-01-21 04:00:00,2980.26,2984.45,2973.26,2977.83
…,…,…,…,…
2026-07-07 19:00:00,1785.59,1791.48,1771.02,1789.59
2026-07-07 20:00:00,1789.59,1792.27,1780.43,1783.32
2026-07-07 21:00:00,1783.31,1788.61,1770.0,1772.84


In [24]:
research.plot_static_timeseries(eth_ts, sym, 'close', time_interval)

## Feature Engineering

In [25]:
eth_ts = research.add_log_return_features(eth_ts, 'close', forecast_horizon, max_no_lags=max_lags)
eth_ts = eth_ts.drop_nulls()
eth_ts

datetime,open,high,low,close,close_log_return,close_log_return_lag_1,close_log_return_lag_2,close_log_return_lag_3,close_log_return_lag_4
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-01-21 05:00:00,2977.84,3001.01,2976.34,2991.71,0.00465,-0.000819,0.006605,-0.001016,0.001803
2026-01-21 06:00:00,2991.71,2992.5,2973.76,2979.01,-0.004254,0.00465,-0.000819,0.006605,-0.001016
2026-01-21 07:00:00,2979.01,2984.42,2964.45,2969.01,-0.003362,-0.004254,0.00465,-0.000819,0.006605
2026-01-21 08:00:00,2969.02,2973.18,2954.15,2964.0,-0.001689,-0.003362,-0.004254,0.00465,-0.000819
2026-01-21 09:00:00,2963.99,2973.62,2947.0,2952.46,-0.003901,-0.001689,-0.003362,-0.004254,0.00465
…,…,…,…,…,…,…,…,…,…
2026-07-07 19:00:00,1785.59,1791.48,1771.02,1789.59,0.002232,-0.01339,0.004575,0.00264,0.004407
2026-07-07 20:00:00,1789.59,1792.27,1780.43,1783.32,-0.00351,0.002232,-0.01339,0.004575,0.00264
2026-07-07 21:00:00,1783.31,1788.61,1770.0,1772.84,-0.005894,-0.00351,0.002232,-0.01339,0.004575


In [26]:
research.plot_distribution(eth_ts, target, no_bins=100)

alt.Chart(...)

In [27]:
research.auto_reg_corr_matrx(eth_ts, target, max_lags)

close_log_return,close_log_return_lag_1,close_log_return_lag_2,close_log_return_lag_3,close_log_return_lag_4
f64,f64,f64,f64,f64
1.0,0.000453,-0.008837,-0.001778,-0.005543
0.000453,1.0,0.000439,-0.008896,-0.001822
-0.008837,0.000439,1.0,0.000438,-0.008806
-0.001778,-0.008896,0.000438,1.0,0.000323
-0.005543,-0.001822,-0.008806,0.000323,1.0


## Benchmark Linear Models

In [28]:
eth_benchmark = research.benchmark_linear_models(
    eth_ts, target, feature_pool, annualized_rate, max_no_features=3, loss=nn.L1Loss()
)
eth_benchmark

features,target,no_trades,win_rate,avg_win,avg_loss,best_trade,worst_trade,ev,std,total_log_return,compound_return,max_drawdown,equity_trough,equity_peak,sharpe,weights,biases
str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""close_log_return_lag_1,close_log_return_lag_4""","""close_log_return""",1007,0.536246,0.004448,-0.004251,0.061749,-0.029609,0.000414,0.006799,0.416791,1.517086,-0.154243,-0.052919,0.512664,5.69728,"""[-0.01031838 -0.01492931]""","""-6.260477675823495e-05"""
"""close_log_return_lag_4""","""close_log_return""",1007,0.512413,0.004429,-0.00428,0.061749,-0.038136,0.000182,0.00681,0.183743,1.201706,-0.17578,-0.000806,0.3334,2.507905,"""[-0.0075405]""","""-7.13812405592762e-05"""
"""close_log_return_lag_1,close_log_return_lag_3""","""close_log_return""",1007,0.532274,0.004261,-0.004465,0.038136,-0.061749,0.000179,0.00681,0.18066,1.198008,-0.149108,0.003219,0.293561,2.465799,"""[-0.02824356 -0.01870682]""","""-7.480748899979517e-05"""
"""close_log_return_lag_1,close_log_return_lag_3,close_log_return_lag_4""","""close_log_return""",1007,0.525323,0.004291,-0.004429,0.038136,-0.061749,0.000152,0.00681,0.15305,1.165383,-0.146241,-0.002191,0.269912,2.088748,"""[-0.02077781 -0.02156897 -0.00667758]""","""-7.410558464471251e-05"""
"""close_log_return_lag_3,close_log_return_lag_4""","""close_log_return""",1007,0.513406,0.004368,-0.004344,0.033773,-0.061749,0.000129,0.006811,0.129716,1.138505,-0.153214,0.003382,0.243429,1.770176,"""[-0.0229476 -0.00231006]""","""-5.3012045100331306e-05"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""close_log_return_lag_1,close_log_return_lag_2,close_log_return_lag_3""","""close_log_return""",1007,0.522344,0.004148,-0.004585,0.038136,-0.061749,-0.000023,0.006812,-0.023334,0.976936,-0.325465,-0.049908,0.275558,-0.318369,"""[-0.02256055 -0.03566389 -0.01544781]""","""-6.24342355877161e-05"""
"""close_log_return_lag_2,close_log_return_lag_3""","""close_log_return""",1007,0.505462,0.004184,-0.004533,0.038136,-0.061749,-0.000127,0.006811,-0.127449,0.880339,-0.3061,-0.152597,0.153504,-1.739224,"""[-0.02717156 -0.01728193]""","""-5.00467540405225e-05"""
"""close_log_return_lag_2,close_log_return_lag_3,close_log_return_lag_4""","""close_log_return""",1007,0.503476,0.004157,-0.004559,0.038136,-0.061749,-0.000171,0.00681,-0.172385,0.841655,-0.390929,-0.205979,0.18495,-2.352786,"""[-0.03157542 -0.01495309 -0.00155121]""","""-5.582797166425735e-05"""


In [29]:
eth_best = eth_benchmark.row(0, named=True)
eth_best_features = eth_best['features'].split(',')
eth_best_features

['close_log_return_lag_1', 'close_log_return_lag_4']

## Best Model: Trade-Level Performance

In [30]:
eth_model = models.LinearModel(len(eth_best_features))
eth_model_trades = research.learn_model_trades(eth_ts, eth_best_features, target, eth_model, loss=nn.L1Loss())
research.plot_column(eth_model_trades, 'equity_curve')

alt.Chart(...)

## Add Transaction Fees

In [31]:
eth_model_trades = research.add_tx_fees_log(eth_model_trades, maker_fee, taker_fee)
research.plot_column(eth_model_trades, 'equity_curve_net_taker')

alt.Chart(...)

## Save Model

In [32]:
torch.save(eth_model.state_dict(), 'ETHUSDT_model_weights.pth')
print(f"Saved ETHUSDT model with features {eth_best_features}")

Saved ETHUSDT model with features ['close_log_return_lag_1', 'close_log_return_lag_4']


## Out-of-Sample Validation (Walk-Forward)

Same walk-forward validation as BTCUSDT: pick features and train fresh on each fold's training window only, evaluate once on that fold's unseen test window, then compare the stitched OOS result against buy-and-hold and a random-signal null distribution.

In [33]:
eth_wf_folds, eth_oos_y_test, eth_oos_y_hat = research.walk_forward_eval(
    eth_ts, target, feature_pool, annualized_rate, n_folds=5, loss=nn.L1Loss()
)
eth_wf_folds

features,target,no_trades,win_rate,avg_win,avg_loss,best_trade,worst_trade,ev,std,total_log_return,compound_return,max_drawdown,equity_trough,equity_peak,sharpe,fold,no_test_obs
str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
"""close_log_return_lag_2,close_log_return_lag_4""","""close_log_return""",402,0.445274,0.003262,-0.002947,0.025672,-0.027106,-0.000183,0.004544,-0.073429,0.929202,-0.15353,-0.099914,0.053617,-3.762406,0,402
"""close_log_return_lag_3,close_log_return_lag_4""","""close_log_return""",402,0.504975,0.00288,-0.002565,0.022635,-0.018429,0.000185,0.004134,0.074211,1.077034,-0.055685,-0.046962,0.08202,4.179004,1,402
"""close_log_return_lag_2""","""close_log_return""",402,0.512438,0.003713,-0.003431,0.038136,-0.032726,0.00023,0.005682,0.092381,1.096783,-0.07137,-0.061661,0.099578,3.785588,2,402
"""close_log_return_lag_1,close_log_return_lag_3""","""close_log_return""",402,0.5199,0.004505,-0.005103,0.035194,-0.029609,-0.000108,0.007238,-0.043307,0.957618,-0.198725,-0.096019,0.102706,-1.393048,3,402
"""close_log_return_lag_1,close_log_return_lag_3""","""close_log_return""",406,0.522167,0.003824,-0.004441,0.021677,-0.061749,-0.000125,0.006633,-0.050736,0.950529,-0.12323,-0.105542,0.017688,-1.76323,4,406


In [34]:
eth_oos_perf = research.eval_model_performance(eth_oos_y_test, eth_oos_y_hat, feature_pool, target, annualized_rate)
eth_oos_perf

{'features': 'close_log_return_lag_1,close_log_return_lag_2,close_log_return_lag_3,close_log_return_lag_4',
 'target': 'close_log_return',
 'no_trades': 2014,
 'win_rate': 0.5009930486593843,
 'avg_win': 0.003652703265253734,
 'avg_loss': -0.0036681168901723776,
 'best_trade': 0.03813611716032028,
 'worst_trade': -0.06174878776073456,
 'ev': -4.368818183821295e-07,
 'std': 0.00576775660738349,
 'total_log_return': -0.0008798742201179266,
 'compound_return': np.float64(0.9991205127556987),
 'max_drawdown': -0.2515544891357422,
 'equity_trough': -0.12039056420326233,
 'equity_peak': 0.19586893916130066,
 'sharpe': np.float64(-0.0070893937631250695)}

### Baselines: Buy & Hold and a Random-Signal Null Distribution

In [35]:
eth_bh_perf = research.buy_and_hold_performance(eth_oos_y_test, annualized_rate)
eth_bh_perf

{'strategy': 'buy_and_hold',
 'no_periods': 2014,
 'total_log_return': -0.276516854763031,
 'compound_return': 0.7584208250045776,
 'std': 0.005764689762145281,
 'max_drawdown': -0.476358562707901,
 'sharpe': -2.2291447850727555}

In [36]:
eth_null_sharpes = research.random_signal_sharpes(eth_oos_y_test, annualized_rate, n_trials=1000, seed=42)
eth_p_value = research.sharpe_p_value(eth_oos_perf['sharpe'], eth_null_sharpes)

print(f"ETHUSDT OOS model Sharpe:      {eth_oos_perf['sharpe']:.2f}")
print(f"ETHUSDT Buy & Hold Sharpe:     {eth_bh_perf['sharpe']:.2f}")
print(f"Random-signal null Sharpe:     mean={eth_null_sharpes.mean():.2f}, std={eth_null_sharpes.std():.2f}")
print(f"P(random signal Sharpe >= model OOS Sharpe): {eth_p_value:.3f}")

ETHUSDT OOS model Sharpe:      -0.01
ETHUSDT Buy & Hold Sharpe:     -2.23
Random-signal null Sharpe:     mean=-0.00, std=2.05
P(random signal Sharpe >= model OOS Sharpe): 0.489


In [37]:
eth_oos_trades = research.model_trade_results(eth_oos_y_test, eth_oos_y_hat)
eth_bh_equity_curve = np.cumsum(eth_oos_y_test.numpy().flatten())

plt.figure(figsize=(12, 6))
plt.plot(eth_oos_trades['equity_curve'], label='Model (walk-forward OOS)')
plt.plot(eth_bh_equity_curve, label='Buy & Hold')
plt.title('ETHUSDT: Out-of-Sample Equity Curve vs Buy & Hold')
plt.xlabel('Period (stitched across folds)')
plt.ylabel('Cumulative Log Return')
plt.legend()
plt.tight_layout()
plt.show()

/var/folders/69/1t3g51kj5y96vspv1zh9f1x40000gn/T/ipykernel_30504/3009361041.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Part 3: BTCUSDT vs ETHUSDT Comparison

In [38]:
comparison = pl.concat([
    btc_benchmark.head(1).with_columns(pl.lit('BTCUSDT').alias('sym')),
    eth_benchmark.head(1).with_columns(pl.lit('ETHUSDT').alias('sym')),
]).select(['sym'] + [c for c in btc_benchmark.columns if c != 'sym'])
comparison

sym,features,target,no_trades,win_rate,avg_win,avg_loss,best_trade,worst_trade,ev,std,total_log_return,compound_return,max_drawdown,equity_trough,equity_peak,sharpe,weights,biases
str,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""BTCUSDT""","""close_log_return_lag_2,close_log_return_lag_4""","""close_log_return""",1013,0.530109,0.003668,-0.003214,0.049699,-0.022936,0.000434,0.005257,0.439939,1.552613,-0.087531,-0.05361,0.442769,7.731603,"""[-0.04638382 -0.02816361]""","""-1.1558848200365901e-05"""
"""ETHUSDT""","""close_log_return_lag_1,close_log_return_lag_4""","""close_log_return""",1007,0.536246,0.004448,-0.004251,0.061749,-0.029609,0.000414,0.006799,0.416791,1.517086,-0.154243,-0.052919,0.512664,5.69728,"""[-0.01031838 -0.01492931]""","""-6.260477675823495e-05"""


In [39]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(btc_model_trades['equity_curve_net_taker'], label='BTCUSDT')
plt.plot(eth_model_trades['equity_curve_net_taker'], label='ETHUSDT')
plt.title('Net (after taker fees) Equity Curve: BTCUSDT vs ETHUSDT')
plt.xlabel('Trade #')
plt.ylabel('Cumulative Log Return')
plt.legend()
plt.tight_layout()
plt.show()

/var/folders/69/1t3g51kj5y96vspv1zh9f1x40000gn/T/ipykernel_30504/3784748504.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Part 4: Statistical Verdict — Is Any of This Actually Useful?

The equity curves and Sharpe ratios earlier in this notebook (the "Benchmark Linear Models" sections) were picked by scanning many feature combinations and keeping the best one — that number alone doesn't prove skill, since the best of many random tries tends to look good even with zero real edge.

This section pulls together the walk-forward *out-of-sample* results for both symbols and checks each one against two bars it has to clear to be called "useful":
1. Beat buy-and-hold (otherwise just holding the asset was better than trading it).
2. Beat the random-signal null distribution by a wide enough margin that the p-value is small (otherwise a coin flip could have produced the same result).

In [40]:
verdict = pl.DataFrame([
    {
        'sym': 'BTCUSDT',
        'oos_sharpe': btc_oos_perf['sharpe'],
        'buy_and_hold_sharpe': btc_bh_perf['sharpe'],
        'beats_buy_and_hold': btc_oos_perf['sharpe'] > btc_bh_perf['sharpe'],
        'random_null_mean_sharpe': btc_null_sharpes.mean(),
        'p_value': btc_p_value,
        'statistically_significant_p<0.05': btc_p_value < 0.05,
    },
    {
        'sym': 'ETHUSDT',
        'oos_sharpe': eth_oos_perf['sharpe'],
        'buy_and_hold_sharpe': eth_bh_perf['sharpe'],
        'beats_buy_and_hold': eth_oos_perf['sharpe'] > eth_bh_perf['sharpe'],
        'random_null_mean_sharpe': eth_null_sharpes.mean(),
        'p_value': eth_p_value,
        'statistically_significant_p<0.05': eth_p_value < 0.05,
    },
])
verdict

sym,oos_sharpe,buy_and_hold_sharpe,beats_buy_and_hold,random_null_mean_sharpe,p_value,statistically_significant_p<0.05
str,f64,f64,i64,f64,f64,bool
"""BTCUSDT""",2.990813,-1.820758,1,-0.065437,0.068,false
"""ETHUSDT""",-0.007089,-2.229145,1,-0.003245,0.489,false


In [41]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, sym_label, null_sharpes, actual_sharpe, p in [
    (axes[0], 'BTCUSDT', btc_null_sharpes, btc_oos_perf['sharpe'], btc_p_value),
    (axes[1], 'ETHUSDT', eth_null_sharpes, eth_oos_perf['sharpe'], eth_p_value),
]:
    ax.hist(null_sharpes, bins=40, alpha=0.7, label='Random-signal null distribution')
    ax.axvline(actual_sharpe, color='red', linewidth=2, label=f'Model OOS Sharpe ({actual_sharpe:.2f})')
    ax.set_title(f'{sym_label}: Model Sharpe vs Chance (p={p:.3f})')
    ax.set_xlabel('Sharpe Ratio')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()

/var/folders/69/1t3g51kj5y96vspv1zh9f1x40000gn/T/ipykernel_30504/637782495.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
